# Análisis desde Jupyter — Madrid Environmental Data

Este notebook lee el fichero curated generado por el workflow **Madrid 3 - Analítica** de n8n desde MinIO,
lo pivota a formato wide y muestra estadísticas básicas.

Ruta en MinIO: `curated/daily_summary/YYYY/MM/DD/YYYY_MM_DD.csv`  
Formato: formato largo (una fila por hora × variable)

In [ ]:
import os
import pandas as pd
from datetime import date, timedelta
from minio import Minio
from io import BytesIO

# Fecha del día anterior (la que genera el workflow de Analítica)
target_date = date.today() - timedelta(days=1)
year  = target_date.strftime('%Y')
month = target_date.strftime('%m')
day   = target_date.strftime('%d')
date_compact = target_date.strftime('%Y_%m_%d')

# Conexión a MinIO
bucket   = os.getenv('MINIO_BUCKET', 'madrid-openmeteo-environment')
endpoint = os.getenv('MINIO_ENDPOINT', 'minio:9000')
client   = Minio(
    endpoint,
    access_key=os.getenv('MINIO_ROOT_USER', 'minioadmin'),
    secret_key=os.getenv('MINIO_ROOT_PASSWORD', 'minioadmin'),
    secure=False
)

# Ruta del CSV curated generado por el workflow n8n Analítica
object_name = f'curated/daily_summary/{year}/{month}/{day}/{date_compact}.csv'
print(f'Leyendo: {object_name}')

response = client.get_object(bucket, object_name)
df_long = pd.read_csv(BytesIO(response.read()))
print(f'Filas leídas: {len(df_long)}')
df_long.head(8)

In [ ]:
# Pivota el formato largo a formato wide (una fila por hora, variables en columnas)
df_wide = df_long.pivot_table(
    index=['date', 'hour', 'time', 'latitude', 'longitude'],
    columns='variable',
    values='value',
    aggfunc='first'
).reset_index()

df_wide.columns.name = None
df_wide = df_wide.sort_values('hour').reset_index(drop=True)
print(f'Forma de la tabla wide: {df_wide.shape}')
df_wide.head()

In [ ]:
# Estadísticas básicas de las variables ambientales
cols = [c for c in ['temperature_2m', 'precipitation', 'ozone', 'carbon_dioxide'] if c in df_wide.columns]
df_wide[cols].describe()

In [ ]:
# Resumen diario por variable (min / avg / max)
resumen = df_long.groupby('variable').agg(
    observaciones=('value', 'count'),
    minimo=('value', 'min'),
    media=('value', 'mean'),
    maximo=('value', 'max')
).round(2)
resumen